# Stratergy

1) Load Data (CSV)
2) eng->sentences->words-> stop words-> tokenization-> OHE->vocabulary->embedding Model-> embeddings
3) RNN Architecture
4) Train
5) Predict

In [4]:
import pandas as pd
import re
from nltk.tokenize import word_tokenize
from torch.nn.functional import embedding

df = pd.read_csv('100_Unique_QA_Dataset.csv')
df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


# Split sentences into wrds after removing punctuations

In [5]:
def tokenize(text):
    text = re.sub(r'[^A-Za-z0-9\w\s]', ' ', text)
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    tokens = word_tokenize(text)
    return tokens

In [6]:
tokenize('Who wrote \'To Kill a Mockingbird\'?')

['who', 'wrote', 'to', 'kill', 'a', 'mockingbird']

# Build a Vocabulary

In [13]:
vocab = {'<UNK>':0}

def build_vocab(row):
    # print(row['question'], row['answer'])
    tokenized_qns = tokenize(row['question'])
    tokenized_ans = tokenize(row['answer'])
    merged_token = tokenized_qns + tokenized_ans
    for token in merged_token:
        if token not in vocab:
            vocab[token] = len(vocab)

df.apply(build_vocab, axis=1)
vocab


{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper': 16,
 'lee': 17,
 'largest': 18,
 'planet': 19,
 'in': 20,
 'our': 21,
 'solar': 22,
 'system': 23,
 'jupiter': 24,
 'boiling': 25,
 'point': 26,
 'water': 27,
 'celsius': 28,
 '100': 29,
 'painted': 30,
 'mona': 31,
 'lisa': 32,
 'leonardo': 33,
 'da': 34,
 'vinci': 35,
 'square': 36,
 'root': 37,
 '64': 38,
 '8': 39,
 'chemical': 40,
 'symbol': 41,
 'for': 42,
 'gold': 43,
 'au': 44,
 'which': 45,
 'year': 46,
 'did': 47,
 'world': 48,
 'war': 49,
 'ii': 50,
 'end': 51,
 '1945': 52,
 'longest': 53,
 'river': 54,
 'nile': 55,
 'japan': 56,
 'tokyo': 57,
 'developed': 58,
 'theory': 59,
 'relativity': 60,
 'albert': 61,
 'einstein': 62,
 'freezing': 63,
 'fahrenheit': 64,
 '32': 65,
 'known': 66,
 'as': 67,
 'red': 68,
 'mars': 69,
 'author': 70,
 '1984': 71,
 'george': 72

# Convert Words into Vocabulary

In [14]:
def text_to_indices(text,vocab):
    indices = []
    for token in tokenize(text):
        if token in vocab:
            indices.append(vocab[token])
        else:
            indices.append(vocab['<UNK>']) #for unknown words
    return indices
text_to_indices("What is france bbb?",vocab)

[1, 2, 6, 0]

In [106]:
from torch.utils.data import Dataset, DataLoader
import torch

class QADataset(Dataset):
    def __init__(self, df, vocab):
        self.df = df
        self.vocab = vocab

    def __len__(self):
        return self.df.shape[0]
    def __getitem__(self, idx):
        q = text_to_indices(self.df.iloc[idx]['question'],self.vocab)
        a = text_to_indices(self.df.iloc[idx]['answer'],self.vocab)
        return torch.tensor(q), torch.tensor(a[0])

In [107]:
dataset = QADataset(df, vocab)
dataset[10]

(tensor([ 1,  2,  3,  4,  5, 56]), tensor(57))

In [108]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [109]:
for question,answer in dataloader:
    print(question,answer)

tensor([[  1,   2,   3,   4,   5, 216]]) tensor([217])
tensor([[ 45, 329,   2,  66,  67,   3, 330,   5, 331]]) tensor([332])
tensor([[ 10, 150,   3, 151, 281,  98, 282,   5,   3, 283]]) tensor([284])
tensor([[  1,   2,   3, 156,  91,  20,  89, 116, 203]]) tensor([204])
tensor([[ 45,  91,  92, 252, 253,  20,  42, 254]]) tensor([255])
tensor([[83, 84, 85, 86, 87, 88, 89]]) tensor([90])
tensor([[ 10,  80, 121]]) tensor([103])
tensor([[ 1,  2,  3, 36, 37,  5, 38]]) tensor([39])
tensor([[ 10, 150,   3, 151, 182,   5,   3,  75, 183]]) tensor([72])
tensor([[ 83,  84, 205,  86,  20,   3, 206, 207, 208]]) tensor([209])
tensor([[ 10,  11, 200, 168, 201]]) tensor([202])
tensor([[ 45, 108,   2,   3,  18]]) tensor([109])
tensor([[  1,   2,   3,   4,   5, 297]]) tensor([298])
tensor([[ 45, 178,   2,   3,  18, 179, 180]]) tensor([181])
tensor([[ 10, 101,   3, 111, 250]]) tensor([251])
tensor([[ 45, 266,   2, 267,  88, 268, 269]]) tensor([270])
tensor([[ 45, 210,   2,  14, 211, 212, 213, 214]]) tensor

In [110]:
len(vocab)

335

# Structure

1) Embeddings (50)
2) Layer1 (Input_Layer)[50 Neurons]Which is same as embedding size
3) Hidden Layer 2 (RNN)[64 Neurons]
4) Layer3 (Output_layer)[335 Neurons]Same as vocab size

In [111]:
import torch.nn as nn

In [112]:
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, 50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.out = nn.Linear(64, vocab_size)

    def forward(self,question):
         embedded_qns = self.embedding(question)
         hidden,final = self.rnn(embedded_qns)
         output = self.out(final.squeeze(0))
         return output

In [113]:
dataset[0][0] # First Question

tensor([1, 2, 3, 4, 5, 6])

In [114]:
x = nn.Embedding(len(vocab), 50)

In [115]:
a=x(dataset[0][0])
print(a)

tensor([[-1.1582e+00, -5.7422e-01, -2.3132e-02, -5.7613e-01,  2.0947e+00,
          1.1952e+00,  1.3523e-01,  8.2972e-01, -3.8172e-01, -7.0541e-01,
          6.3587e-01, -2.5433e-01,  2.6600e+00, -1.0779e+00, -5.8763e-01,
          2.8882e+00, -7.4042e-01,  6.7431e-01, -1.4110e+00, -1.1447e+00,
          7.4409e-01, -1.2444e+00, -7.1768e-01, -1.3831e-01,  7.0566e-01,
          2.4235e+00, -9.0634e-01,  1.4564e+00, -1.9971e-01,  2.6596e+00,
         -1.3687e+00,  1.5510e+00, -1.5812e+00, -3.6941e-01, -8.5824e-01,
         -6.7909e-01,  3.8126e-01, -3.3130e-01,  8.0926e-01,  1.5855e+00,
         -9.2923e-01,  2.7694e+00, -9.0042e-01,  1.1838e+00,  1.0203e+00,
          9.6338e-01, -2.8050e-01,  6.2151e-01, -1.0708e+00,  1.3482e+00],
        [-3.5320e-01, -6.4356e-01,  1.7495e-01, -1.8748e+00,  6.9255e-02,
         -1.1690e+00, -8.7721e-01,  5.4689e-01,  2.7966e-01,  3.5379e-01,
          1.2576e+00, -3.7502e-01,  7.2350e-01, -1.4708e+00, -2.0502e+00,
          7.1983e-01,  1.3684e+00,  5

In [116]:
a.shape

torch.Size([6, 50])

In Above Code Each word is converted to a embedding of 50..therefore 6wordsx50Embeddings

In Code below we try and understand the outputs of RNN


In [117]:
r = nn.RNN(50, 64)
print(r(a))

(tensor([[-0.1296, -0.6887, -0.4839, -0.3785, -0.4272,  0.2950, -0.7642, -0.4288,
         -0.3318, -0.5165,  0.0937,  0.2822,  0.3370,  0.6056, -0.4256, -0.7756,
         -0.6581, -0.8050,  0.1680,  0.5530, -0.5107,  0.0918,  0.3735,  0.5301,
         -0.7573,  0.0650,  0.1959,  0.3416,  0.6253,  0.0224, -0.1122,  0.7113,
         -0.0899, -0.2746, -0.7109,  0.2082,  0.0299, -0.7090,  0.4188, -0.3114,
         -0.6270,  0.3577, -0.4368,  0.3089, -0.7870, -0.4071,  0.1255, -0.2796,
          0.1246, -0.1653, -0.1912, -0.5109,  0.0248,  0.7297, -0.7096,  0.4599,
         -0.2753,  0.1383,  0.2357, -0.1690,  0.3393, -0.1603, -0.4032,  0.2872],
        [-0.2224, -0.2844,  0.3799,  0.3270, -0.1845, -0.0285, -0.5864, -0.0104,
          0.0342,  0.4478, -0.0917, -0.3767,  0.5121,  0.4051, -0.3195,  0.2021,
         -0.3960, -0.7160,  0.1513,  0.3056, -0.8077,  0.6741,  0.2720, -0.5576,
         -0.8445, -0.4840,  0.1891,  0.6944,  0.0386,  0.7454,  0.6393,  0.4182,
         -0.5835, -0.9408,

In [118]:
print(len(r(a)))
print(r(a)[0].shape) # Output of Individual Later of RNN O1,O2,O3,O4,O5,O6
print(r(a)[1].shape) # Final output of RNN

2
torch.Size([6, 64])
torch.Size([1, 64])


In [119]:
z = nn.Linear(64, len(vocab))
ans = z(r(a)[1])
print(ans)

tensor([[-1.4670e-01,  3.8818e-01,  4.3718e-01, -1.5203e-01,  9.1401e-02,
         -2.6185e-01, -1.5172e-01,  2.2631e-01,  2.0842e-01,  6.6782e-02,
          1.1875e-01,  1.6169e-01,  1.4095e-01, -2.7222e-01,  8.0166e-02,
         -2.1111e-01, -1.3303e-01, -4.8778e-01, -3.5757e-02, -5.8869e-01,
         -2.1505e-01, -1.9113e-01, -2.1618e-01,  7.0156e-02, -7.5633e-01,
          3.6349e-03, -2.4739e-02,  2.8577e-02,  1.9786e-01, -2.4580e-01,
          1.3774e-01, -2.3455e-01,  1.1189e-02,  1.4209e-01,  1.3710e-01,
          2.3027e-01, -2.4295e-01, -4.5828e-01, -3.5426e-01, -2.3228e-01,
          3.2710e-02, -6.5541e-01, -3.9951e-01,  2.8419e-01, -3.2960e-01,
         -3.2340e-01,  1.2385e-01,  1.8736e-01, -2.1071e-01,  1.3593e-01,
          3.1262e-01,  4.5732e-01,  1.3620e-01,  2.1790e-01, -2.6747e-01,
          5.3808e-01,  3.7486e-01,  4.1608e-01, -1.4054e-02,  4.9503e-01,
         -5.0563e-02, -1.5086e-01, -1.3678e-01, -2.5669e-01,  2.2176e-02,
         -5.3888e-01, -1.3971e-01, -1.

In [120]:
print(ans.argmax(dim=1))
vocab['da'] # output predicted by random weights

tensor([304])


34

# Since RNN gives multiple outputs, we cant use Sequential Later in out Model, because Sequential layer assumes that each layer generate one output and pass it onto the next layer

In [121]:
learning_rate = 0.001
epochs = 20

In [122]:
model = SimpleRNN(len(vocab))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [125]:
# training loop
for epoch in range(epochs):
    total_loss = 0
    for qns,ans in dataloader:
        optimizer.zero_grad()
        outputs = model(qns)
        loss = criterion(outputs, ans)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epochs: {epoch}, Loss: {total_loss:4f}")

Epochs: 0, Loss: 9.238751
Epochs: 1, Loss: 8.138873
Epochs: 2, Loss: 7.246276
Epochs: 3, Loss: 6.434875
Epochs: 4, Loss: 5.765920
Epochs: 5, Loss: 5.212904
Epochs: 6, Loss: 4.720024
Epochs: 7, Loss: 4.293029
Epochs: 8, Loss: 3.920892
Epochs: 9, Loss: 3.588384
Epochs: 10, Loss: 3.306474
Epochs: 11, Loss: 3.051542
Epochs: 12, Loss: 2.805305
Epochs: 13, Loss: 2.597494
Epochs: 14, Loss: 2.407396
Epochs: 15, Loss: 2.235932
Epochs: 16, Loss: 2.083716
Epochs: 17, Loss: 1.941960
Epochs: 18, Loss: 1.816356
Epochs: 19, Loss: 1.693445


In [151]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know \"" + list(vocab.keys())[index] + "\" maybe?")
    return

  print(list(vocab.keys())[index])

In [160]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [153]:
list(vocab.keys())[7]

'paris'

In [154]:
predict(model, "what is the em?")

I don't know "margaretthatcher" maybe?
